# Time optimization of the app development process

IS6003 final assignment. Research method: DSRM (Peffers et al., 2007). Process analysis follows *Fundamentals of BPM*, 2nd ed. (Dumas et al.). Page numbers are the book's printed page numbers.

This notebook runs the whole study from start to end with one seed:

1. check the models against the book's modeling rules;
2. simulate the corrected as-is model and write the event log;
3. measure the baseline and cross-check it with the book's flow analysis;
4. load the value-added classification;
5. train the ML models and rank the drivers of delay and rework;
6. compare those drivers with the effects we planted in the simulator;
7. load the redesign: the change catalog and the to-be models;
8. simulate the as-is process and every combination of changes, and read off the saving and the ablation;
9. write `web/results.json` for the page.

All logic lives in the `bpmopt` package and is tested there. The cells below only call it and show the output. A full run takes about 8 minutes, most of it in step 8.

In [ ]:
from pathlib import Path

import pandas as pd

from bpmopt.analysis import baseline_metrics, flow_analysis_check
from bpmopt.bpmn import read_bpmn
from bpmopt.event_log import export_csv
from bpmopt.features import case_features
from bpmopt.lint import lint
from bpmopt.ml import cycle_time_task, ml_section, rework_tasks
from bpmopt.parameters import assumptions, load_parameters
from bpmopt.planted import load_planted_effects
from bpmopt.results import build_results, element_names, export_results
from bpmopt.scenarios import ablation, headline, load_catalog, run_scenarios
from bpmopt.simulation import simulate
from bpmopt.value_added import load_value_added, time_per_class

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ORIGINAL = ROOT / "current_process.bpmn"
MODEL = ROOT / "models/as_is.bpmn"
PARAMETERS = ROOT / "models/as_is.params.toml"
VALUE_ADDED = ROOT / "models/as_is.value_added.toml"
PLANTED = ROOT / "models/as_is.planted_effects.toml"
CATALOG = ROOT / "models/to_be.catalog.toml"
TO_BE = ROOT / "models/to_be.bpmn"  # the full to-be model, C1 and C2 (C3 and C4 only change parameters)
APPROVED = "End_ProjectClosed"  # end event of an approved project
LOG_CSV = ROOT / "notebooks/output/as_is_event_log.csv"
SEED = 42
N_CASES = 2_000  # cases kept after the warm-up, in the baseline run and in every scenario run
REPLICATIONS = 30  # runs per scenario

pd.set_option("display.precision", 2)

## 1. Check the models

The linter applies the rules from §5.4 that a program can check: the structural rules of §5.4.1 (pp.183-187), the labeling rules of §5.4.3 (pp.189-190) and the seven guidelines of 7PMG in §5.4.4 (pp.192-193).

The original model, `current_process.bpmn`, is kept unchanged for reference. The corrected model, `models/as_is.bpmn`, is what we simulate. Its changes are listed in `models/as_is_changelog.md`. The original should show violations and the corrected one none.

In [ ]:
original = pd.DataFrame(lint(ORIGINAL))
print(f"current_process.bpmn: {len(original)} violations")
original.groupby(["rule", "severity"]).size().rename("count").to_frame()

In [ ]:
as_is_violations = lint(MODEL)
print(f"models/as_is.bpmn: {len(as_is_violations)} violations")

## 2. Parameters

The simulator needs an arrival rate, case attributes, task durations, branch probabilities and lane capacities. We have no real data, so every value is an estimate for a bank IT division, marked "Proposed default" in `models/as_is.params.toml`. Entries marked "Student" carry the student's answers: outright rejection of a proposal is rare (3% of decisions), and the cycle time ends at go-live. The book warns that simulation inputs should come from observation where possible and be checked with the people who run the process (§7.3.4, p.287). This is the main threat to validity, so the whole table goes on the page.

Requests arrive at 0.234 per working day, a bit under one a week (arrival rate, p.266). That rate keeps the busiest lane, QC, at about 87% busy, so the results aren't dominated by a queue close to saturation (§7.2, p.276). The first 250 working days are a warm-up and are dropped, so the queues don't start empty.

In [ ]:
model = read_bpmn(MODEL)
params = load_parameters(PARAMETERS, model=model)
print(f"Parameter version: {params.version}. Cycle time ends at the end of {params.cycle_time_end}.")
pd.DataFrame(assumptions(params))

## 3. Simulate and write the event log

The simulator runs the BPMN file itself, so the diagram is exactly the process that is simulated. Each lane is a pool of identical resources with the capacity above, so cases queue when a lane is busy.

The log has one row per element execution, with the columns of the book's event log: case ID, activity, resource (here the lane) and time (§11.3, p.422, Fig. 11.4 on p.423). Instead of one timestamp we keep ready, start and end times, because the book needs start and end times to tell processing time from waiting time (§11.5, pp.443-444). The case attributes come last, as extra data attributes (p.422). XOR-split decisions are zero-length rows with the flow taken, so rework can be counted.

The CSV goes to `notebooks/output/`, which is not committed. The notebook rewrites it on every run.

In [ ]:
log = simulate(model, params, n_cases=N_CASES, seed=SEED)
LOG_CSV.parent.mkdir(exist_ok=True)
export_csv(log, LOG_CSV)
print(f"{log.case_id.nunique():,} cases, {len(log):,} rows, written to {LOG_CSV.relative_to(ROOT)}")
log.head(8)

## 4. Baseline

From the event log:

- Cycle time (p.256) runs from the request to go-live, the end of "Start live operation" (why, see below). A case that ends earlier, at "Proposal rejected", ends at that end event.
- Processing time is the time in which some task of the case is in progress, up to go-live. Parallel work counts once, as in the book's theoretical cycle time (p.262).
- Waiting time is the cycle time minus processing time, the time "where no work is being done to advance the process" (p.261).
- Cycle time efficiency is the mean processing time over mean cycle time (Eq. 7.5, p.262).

Why go-live: the business unit has its application once it is live. "Monitor live operation" is about two weeks of hypercare and "Evaluate project results" is the closing report. Both stay in the model and the log, but no redesign aims at them, so they would only dilute the saving. The cut-off is one entry in the parameter file (`[cycle_time]`), and the baseline, the flow analysis check, the ML and the scenario runner all read it from there.

The headline counts every case, rejected proposals included. Approved projects (cases that end at "Project closed") are shown next to it as a check.

In [ ]:
baseline = baseline_metrics(log, model, ends_at=params.cycle_time_end)
pd.DataFrame({k: baseline[k] for k in ("cycle_time", "processing_time", "waiting_time")}).T

In [ ]:
names = element_names(model)

print(f"Cycle time efficiency: {baseline['cycle_time_efficiency']:.2f}")
print(f"Waiting share of the mean cycle time: {baseline['waiting_time']['mean'] / baseline['cycle_time']['mean']:.0%}")
pd.DataFrame(
    {names[o["end_event_id"]]: {"cases": o["case_count"], **o["cycle_time"], "efficiency": o["cycle_time_efficiency"]} for o in baseline["by_outcome"]}
).T

Cycle time efficiency is high for a real process. The book's Table 7.4 (p.263) reports values around 5% or lower. Ours is high because the simulation only has queueing for lanes with a few identical resources. It has no handoff delays, batching, holidays or people working on several projects at once, which are where most waiting comes from in practice (p.261). Read this as a lower bound on waiting.

### Rework and waiting per activity

Rework rate per rework gateway: the share of the cases that reached the gateway and looped back at least once. Rework loops are defect waste (§6.2, p.222) and lengthen the cycle time by a factor of 1 / (1 - r) (Eq. 7.4, p.260). Gateways in the model are unnamed, so `element_names` names each one after the task just before it, the same names the page shows.

In [ ]:
rework = pd.DataFrame(baseline["rework"])
rework.insert(1, "decision", rework.gateway_id.map(names))
rework

Mean waiting per activity run, from ready to start, longest first. This is where cases queue for a free resource (waiting waste, §6.2, pp.221-222).

In [ ]:
waiting = pd.DataFrame(baseline["waiting_by_activity"])
waiting.insert(1, "activity", waiting.activity_id.map(names))
waiting

## 5. Flow analysis check

The book computes the theoretical cycle time from the model and the mean task durations with four block formulas: sequence, XOR, AND and rework (§7.1.1, pp.256-260). Flow analysis ignores resource contention (§7.1.7, p.273), so we compare it with a second simulation that has unlimited capacity in every lane, with the same seed and case count. Both stop at go-live, and a rejected case at its end event, so they cover the same span. The two should agree within sampling error. Branch probabilities that depend on case attributes are averaged over attribute draws, which is our extension of the book's formulas.

In [ ]:
check = flow_analysis_check(model, params, n_cases=N_CASES, seed=SEED)
print(f"Book formula: {check['theoretical']:.1f} days. Simulated with unlimited capacity: {check['simulated']:.1f} days. "
      f"Difference: {check['simulated'] / check['theoretical'] - 1:+.1%}")

## 6. Value-added classification

Each task gets the class of its main step: value-adding (VA), business value-adding (BVA) or non-value-adding (NVA), as defined on pp.214-215. The classes are a draft awaiting the student's review (`docs/analysis/phase1-qualitative.md`). The table below adds how much task time each class takes on the simulated log, per case, up to go-live.

In [ ]:
value_added = load_value_added(VALUE_ADDED, model)
time_per_class(log, value_added, ends_at=params.cycle_time_end)

## 7. Machine learning

The book has no chapter on predictive ML. The method is ours and is set out in the spec:

- Features are the case attributes drawn at arrival, the only things known when a request comes in. One row per case.
- The split follows case start time. The last 20% of cases are the test set. Training uses the first 80%, minus any case that was still open when the first test case started, so no model learns from later cases.
- The models are a reference that ignores the inputs (mean, or majority class), a linear or logistic regression, and gradient boosting.
- Metrics are MAE and R² for cycle time, ROC-AUC and F1 for rework.
- Drivers are ranked by the mean absolute SHAP value of the gradient boosting model on the test cases, one value per attribute.

Task A predicts the cycle time, up to go-live, and only for approved projects: it trains and tests on cases that end at "Project closed". A rejected proposal ends early for a reason no attribute explains (the split is fixed), so mixing those cases in would mostly add noise. Task B has one classifier per rework gateway and predicts whether a case loops back there at least once. Only cases that reached the gateway count.

In [ ]:
case_features(log).head()

In [ ]:
tasks = [
    cycle_time_task(log, seed=SEED, ends_at=params.cycle_time_end, outcome=APPROVED),
    *rework_tasks(log, model, seed=SEED),
]


def title(task):
    return "cycle time" if task["kind"] == "regression" else f"rework, {names[task['gateway_id']]}"


fitted = [t for t in tasks if "skipped" not in t]
pd.DataFrame({m["id"]: m["metrics"] for t in fitted if t["kind"] == "regression" for m in t["models"]}).T

In [ ]:
pd.DataFrame({(title(t), m["id"]): m["metrics"] for t in fitted if t["kind"] == "classification" for m in t["models"]}).T

A rework gateway where the training or the test cases all went the same way can't be fitted or scored. The package then skips that gateway and records why, instead of stopping the run. The page shows the reason in place of the scores.

In [ ]:
skipped = {t["id"]: t["skipped"] for t in tasks if "skipped" in t}
print(skipped or "No gateway skipped with this seed.")

Drivers per task, ranked by mean |SHAP|, rank 1 = strongest. For cycle time the values are in days, for rework in log-odds.

In [ ]:
drivers = pd.DataFrame({title(t): {d["feature"]: d["mean_abs_shap"] for d in t["drivers"]} for t in fitted})
ranks = drivers.rank(ascending=False).astype(int)
drivers.round(2).astype(str) + "  (#" + ranks.astype(str) + ")"

## 8. ML drivers against the planted effects

The log is synthetic, so we know the truth. `models/as_is.planted_effects.toml` lists every attribute effect put into the parameters, and `models/as_is_planted_effects.md` explains each one. Loading the file checks it against the parameters, and the package works out from it which attributes each ML task should find. `requesting_unit` affects nothing and is the negative control: if the ML ranks it high, the ranking is picking up noise. The table puts each task's planted attributes next to the ranks the ML gave them.

In [ ]:
planted = load_planted_effects(PLANTED, params)
expected = {title(t): planted.expected_drivers(model)[t["id"]] for t in fitted}
control = planted.negative_control

pd.DataFrame(
    {
        "planted": {task: ", ".join(attrs) or "(nothing)" for task, attrs in expected.items()},
        "their ranks": {task: ", ".join(f"#{ranks.loc[a, task]}" for a in attrs) or "-" for task, attrs in expected.items()},
        "top 3 found": {task: ", ".join(ranks[task].sort_values().index[:3]) for task in expected},
        f"{control} rank": {task: f"#{ranks.loc[control, task]} of {len(ranks)}" for task in expected},
    }
)

What this shows with seed 42 and 2,000 cases. The cycle time task uses the 1,924 approved projects: 385 test cases, and 1,522 training cases, since 17 of the first 1,539 were still open when the test cases started. The rework tasks use every case that reached the gateway.

- For cycle time, the drivers come out in the planted order of strength: complexity, security level, integrations, then team workload and project type. The negative control, `requesting_unit`, ranks last.
- At the test loop the top three drivers are exactly the three planted ones: complexity, security level and integrations. At the plan loop, project type comes first, which is the one planted effect there.
- At the release loop project type comes second, behind team workload, which has no effect there. Only about 8% of releases are rejected, so the test set has few positive cases.
- The negative control never ranks first. It is last for cycle time and at the test loop, third at the plan loop and fourth at the release and proposal loops. At the proposal loop nothing is planted, all attributes get small values and the order among them means little.
- The cycle time models beat predicting the mean, but only just: R² is about 0.1 for both on the test cases (the reference scores 0), and the MAE drops from 31.8 to about 30.1 days. Most of a project's cycle time comes from chance: how many times work loops back and how long the queues are. No attribute known at the start explains that. Even a linear fit scored on the same approved projects it was fitted on explains only about 15% of the variation (`models/as_is_planted_effects.md`).
- Keeping the rejected proposals out helps. They end early for a reason no attribute explains, since the proposal decision has fixed probabilities. In the first run (all cases, monitoring counted, 10% rejected) R² was about 0.
- The rework models clearly beat the reference only at the test loop (ROC-AUC about 0.63) and, for the logistic model, at the release loop (0.69). The plan loop scores about 0.55, and the proposal loop, where nothing is planted, about 0.5.
- Team workload is the only continuous attribute. Trees can split it in many places, so it picks up some SHAP value even where it has no effect. Small SHAP values for it shouldn't be read as a driver.

This is a threat to validity. It's a check of the method, not a finding about the real process. The ML can only find effects we put in, and it finds the strong ones. The weak effects (project type and team workload on cycle time) rank above the negative control here, but not by much, and with 2,000 cases another seed could swap them. On real data, a driver that ranks no higher than an attribute known to have no effect shouldn't be used to justify a redesign. For the redesign, the evidence that holds is complexity, security level and integrations in the cycle time and at the test loop, and project type at the plan loop.

## 9. Redesign

The redesign is a human decision (Phase 5). The student delegated the design to Claude and approved all four changes on 2026-11-03. Each change targets a finding above and applies a heuristic from Appendix A. Its trade-off is stated on the Devil's Quadrangle (§8.1.3, p.303). Why each change exists and what it costs is written up in `models/to_be_changelog.md`.

- **C1** runs the four QC checks in parallel (Parallelism, A.3). QC is the main queue.
- **C2** retests only what was fixed instead of running all four checks again (Activity elimination, A.2). The test loop has the highest rework rate.
- **C3** gives complex and highly classified cases a longer requirements step, with fewer defects later (Triage, A.2). Complexity and security level are the two strongest ML drivers.
- **C4** adds one QC tester (Extra resources, A.4).

C1 and C2 change the model, so each combination of them has its own BPMN file. C3 and C4 only change parameters. The catalog file says which model and which parameter overrides each change brings. Loading it resolves all 16 combinations and checks each one against its model, so a broken one fails here, before any simulation.

In [ ]:
catalog = load_catalog(CATALOG, model=model, params=params)
pd.DataFrame(catalog.entries()).set_index("id")[["title", "driver", "heuristic", "trade_off"]]

The to-be models get the same rule check as the as-is model. They have zero errors. Each has one warning: with 31 to 33 flow nodes they are just over the 30 elements of 7PMG G7 (§5.4.4, p.193). The book's fix is a sub-process, which our simulator doesn't support, so the student accepted the warning. It goes into the threats to validity.

In [ ]:
for file in sorted((ROOT / "models").glob("to_be*.bpmn")):
    violations = lint(file)
    print(f"{file.name}: {len(read_bpmn(file).nodes)} flow nodes, " + (", ".join(f"{v.rule} ({v.severity})" for v in violations) or "clean"))

## 10. Evaluate the redesign

The scenario runner simulates the as-is process and all 2^4 = 16 combinations of changes. The set-up is the same as the baseline run: the same parameter file, the same 2,000 cases per run after the same 250-day warm-up, and seeds derived from the same seed (42). Each scenario runs 30 times. Run r of every scenario uses the same seed, so every scenario sees the same arrivals, case attributes and draws per task and gateway. This is the method of common random numbers: differences between scenarios come from the design, not from luck. The book doesn't cover warm-up periods, replications or common random numbers; they are standard practice in discrete-event simulation, and the spec asks for them.

For each scenario we report the mean, median and 90th percentile cycle time, averaged over the 30 runs, with 95% t-intervals. The saving is the as-is mean minus the scenario's mean, paired by run, so its interval only reflects how the difference varies. The saving as a share of the as-is mean is paired the same way.

This cell takes about 6 minutes.

In [ ]:
scenarios = run_scenarios(catalog, n_cases=N_CASES, seed=SEED, replications=REPLICATIONS)


def label(changes):
    return "+".join(changes) or "as-is"


pd.DataFrame(
    {
        label(s["changes"]): {
            "mean": s["cycle_time"]["mean"]["value"],
            "median": s["cycle_time"]["median"]["value"],
            "P90": s["cycle_time"]["p90"]["value"],
            "saving (days)": s["saving"]["value"],
            "saving CI": f"[{s['saving']['ci_low']:.1f}, {s['saving']['ci_high']:.1f}]",
            "faster": f"{s['saving_share']['value']:.1%}",
            "faster CI": f"[{s['saving_share']['ci_low']:.1%}, {s['saving_share']['ci_high']:.1%}]",
        }
        for s in sorted(scenarios, key=lambda s: -s["saving"]["value"])
    }
).T

### The baseline is one of many possible runs

The baseline in section 4 is one run with seed 42. The as-is scenario here averages 30 runs of the same process with the same settings, so the two means differ. The gap is run-to-run variation: QC is busy about 87% of the time, and a run that happens to get a few complex cases close together builds a long queue that lifts its whole mean (§7.2, p.276, on why waiting grows fast as utilisation nears 100%). The baseline stays the right source for *where* time goes. The evaluation is the right source for *how much* a change saves.

In [ ]:
as_is_runs = pd.Series(next(s for s in scenarios if not s["changes"])["replication_means"])
print(f"Baseline run (seed {SEED}): {baseline['cycle_time']['mean']:.1f} days")
print(f"As-is over {REPLICATIONS} runs: {as_is_runs.mean():.1f} days, runs from {as_is_runs.min():.1f} to {as_is_runs.max():.1f}, "
      f"the baseline run is lower than {(as_is_runs > baseline['cycle_time']['mean']).mean():.0%} of them")
as_is_runs.describe().to_frame("as-is mean per run").T

### Ablation

The ablation reads two views off the 16 scenarios:

- **alone**: the saving of the scenario with only that change;
- **full minus**: how many days longer the full design gets when that change is dropped. A negative value means the full design is faster without it. Its interval pairs the two scenarios run by run.

When changes don't overlap, the two views agree. When they target the same queue, a change saves less on top of the others than it does alone.

In [ ]:
ablation_rows = ablation(scenarios)
pd.DataFrame(
    {
        row["change"]: {
            "alone": row["alone"]["value"],
            "alone CI": f"[{row['alone']['ci_low']:.1f}, {row['alone']['ci_high']:.1f}]",
            "full minus": row["full_minus"]["value"],
            "full minus CI": f"[{row['full_minus']['ci_low']:.1f}, {row['full_minus']['ci_high']:.1f}]",
        }
        for row in ablation_rows
    }
).T

In [ ]:
summary = headline(scenarios, ablation_rows)
full = next(s for s in scenarios if set(s["changes"]) == set(summary["changes"]))
best = next(s for s in scenarios if set(s["changes"]) == set(summary["best"]))
share = full["saving_share"]
print(f"Full design {label(summary['changes'])}: {share['value']:.1%} faster (95% CI {share['ci_low']:.1%} to {share['ci_high']:.1%}), "
      f"{full['saving']['value']:.1f} days")
print(f"Contributes most to the full design: {summary['top_change']}")
print(f"Best combination: {label(summary['best'])}, {best['saving_share']['value']:.1%} faster, {best['saving']['value']:.1f} days")

### What the evaluation shows

With seed 42, 2,000 cases per run and 30 runs per scenario:

- **Headline.** The full design (all four changes) is 25.3% faster than the as-is process (95% CI 23.7% to 27.0%). The mean cycle time drops from 140.8 to 104.5 working days, a saving of 36.3 days (CI 32.0 to 40.5). The whole interval is above the 20% target from Phase 1, so the objective is met.
- **Which change contributes most.** In the full design, C1: without parallel checks the design takes 11.3 days longer (CI 11.1 to 11.5). On its own, C2 saves the most (27.5 days against 20.4 for C1). The two views disagree because C1 and C2 work on the same QC queue. Once one of them has shortened it, the other has less left to remove. The headline names C1, because the question is what the proposed design needs.
- **C3 hurts once C1 and C2 are on.** Alone it saves 15.6 days. In the full design it costs 2.8 days (CI -3.2 to -2.4), clearly below zero. With the QC queue drained (QC was busy about half the time in the full design in ticket 15's check), fewer defects save little, while the longer requirements step still loads the business unit lane (from about 75% to 83% busy in the same check).
- **C4 adds little on top.** One more tester saves 13.0 days alone but only 0.5 days in the full design, for the cost of a salary.
- **The best combination isn't the full design.** C1+C2+C4 is 27.4% faster (CI 25.8% to 28.9%), and C1+C2 alone is 26.9%. We still report the full design as the headline, because it is the design the student approved. Reporting the best of 16 combinations, picked after looking at all of them, would overstate the gain.
- **The changes only partly add up.** The four savings alone sum to about 77 days, against 36 for all four together. Three of the four target the same queue, so the saving of each depends on what else changed. That is why the ablation looks at the full design, not only at each change alone.
- **Savings are precise, cycle times less so.** The as-is mean has a CI of 135.3 to 146.3 days, but C1's saving has 19.4 to 21.4. Common random numbers make the noise mostly cancel out in a paired difference.

Dropping C3 is a real option for the student: it would trade the fewer defects C3 aims for against about 3 days. The simulation can't settle that, because it counts a defect only by the rework time it causes.


## 11. Write the results for the page

`build_results` collects the run settings, page text, assumptions, threats to validity, baseline and flow analysis check. The value-added classes, the ML section, the change catalog, the scenarios, the ablation and the headline are added here. `export_results` copies both models next to the results, lints them, checks everything against the JSON schema and writes `web/results.json`.

The threats to validity are page text, in `src/bpmopt/content/page.en.toml`: the data is synthetic, the parameters and the effects of the changes are estimates, there is no working calendar, resources are identical, QC runs close to its limit, and the to-be models break G7.

In [ ]:
results = build_results(log=log, model=model, params=params, seed=SEED, replications=REPLICATIONS)
results["value_added"] = value_added
results["ml"] = ml_section(tasks)
results["changes"] = catalog.entries()
results["scenarios"] = scenarios
results["ablation"] = ablation_rows
results["headline"] = summary
path = export_results(results, ROOT / "web", models={"as_is": MODEL, "to_be": TO_BE})
print(f"Wrote {path.relative_to(ROOT)}")